# External validation with known isoform-specific PPIs

We compare Splitpea’s rewired network built using rMATS data from the PC3E-GS689 dataset against IntAct's currated cancer interaction set. The network we use comes from the tutorial in /examples/condition_mode.ipynb in the rMATS example section. Please see that tutorial for details on how the network was built. Here we use the PC3E_GS689.edges.pickle file that was generated in that tutorial.

In [ ]:
import pandas as pd
import re 
import pickle 

In [ ]:
# Download the IntAct cancer interaction set 
!wget https://ftp.ebi.ac.uk/pub/databases/intact/current/psimitab/datasets/Cancer.zip
!unzip Cancer.zip

# additional mapping file for organism id conversions
!wget https://raw.githubusercontent.com/ylaboratory/splitpea/master/reference/hsa_mapping_all.txt

In [ ]:
path = "Cancer.txt"
df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)

## Get isoform-specific PPIs from IntAct:

In [ ]:
UNIPROT_TOKEN_RE = re.compile(r"^uniprotkb:([A-Z0-9]{6,10}(?:-\d+)?)$", re.I)
ISOFORM_RE = re.compile(r"^[A-Z0-9]{6,10}-\d+$", re.I)

def edges_to_df(G, label: str):
    edges = list(G.edges())
    df = pd.DataFrame(edges, columns=["entrezA", "entrezB"])

    df["entrezA"] = pd.to_numeric(df["entrezA"], errors="ignore")
    df["entrezB"] = pd.to_numeric(df["entrezB"], errors="ignore")

    a = df["entrezA"]
    b = df["entrezB"]
    df["entrez1"] = a.where(a <= b, b)
    df["entrez2"] = b.where(a <= b, a)

    df = df[["entrez1", "entrez2"]].drop_duplicates()
    df[label] = True
    return df

def first_uniprot_id(field: str):
    if not isinstance(field, str) or not field:
        return None
    for tok in field.split("|"):
        tok = tok.strip()
        m = UNIPROT_TOKEN_RE.match(tok)
        if m:
            return m.group(1)
    return None

def is_isoform_acc(acc: str):
    return isinstance(acc, str) and bool(ISOFORM_RE.match(acc))

GENE_NAME_RE = re.compile(r"^(?:uniprotkb:)?([A-Za-z0-9_.-]+)\(gene name\)$", re.I)

def gene_from_aliases(alias_field: str):
    if not isinstance(alias_field, str) or not alias_field:
        return None
    for tok in alias_field.split("|"):
        tok = tok.strip()
        if "(gene name)" in tok.lower():
            m = GENE_NAME_RE.match(tok)
            if m:
                return m.group(1)
            base = tok.split("(")[0]
            if ":" in base:
                base = base.split(":", 1)[1]
            base = base.strip()
            if base:
                return base
    return None

df["uniprotA"] = df["#ID(s) interactor A"].map(first_uniprot_id)
df["uniprotB"] = df["ID(s) interactor B"].map(first_uniprot_id)

iso_mask = df["uniprotA"].map(is_isoform_acc) | df["uniprotB"].map(is_isoform_acc)
df_iso = df.loc[iso_mask].copy()

df_iso["geneA"] = df_iso["Alias(es) interactor A"].map(gene_from_aliases)
df_iso["geneB"] = df_iso["Alias(es) interactor B"].map(gene_from_aliases)

map_path = "hsa_mapping_all.txt"
hsa_map = pd.read_csv(map_path, sep="\t", dtype=str)
sym2entrez = hsa_map[["symbol", "entrez"]].copy()

sym2entrez["symbol"] = sym2entrez["symbol"].astype(str).str.strip()
sym2entrez["entrez"] = sym2entrez["entrez"].astype(str).str.strip()

sym2entrez = sym2entrez[sym2entrez["symbol"].ne("") & sym2entrez["symbol"].notna()].copy()

sym2entrez = sym2entrez.drop_duplicates(subset=["symbol"], keep="first").copy()
df_out = df_iso.copy()

for col in ["geneA", "geneB"]:
    df_out[col] = df_out[col].astype(str).str.strip()
    df_out.loc[df_out[col].isin(["", "nan", "None"]), col] = pd.NA

df_out = df_out.merge(
    sym2entrez.rename(columns={"symbol": "geneA", "entrez": "entrezA"}),
    on="geneA",
    how="left"
)

df_out = df_out.merge(
    sym2entrez.rename(columns={"symbol": "geneB", "entrez": "entrezB"}),
    on="geneB",
    how="left"
)

## Load Splitpea network 

In [ ]:
# replace with path to the saved Splitpea prostate cancer network from tutorial (/examples/condition_mode.ipynb)
with open("PC3E_GS689.edges.pickle", "rb") as f:
    prostate_splitpea = pickle.load(f)
    
df_all = edges_to_df(prostate_splitpea, "in_splitpea")

df_all = df_all.rename(columns={"entrez1": "entrezA", "entrez2": "entrezB"})

## Standarize edges for comparision 

In [ ]:
def canonicalize_edges(df, a="entrezA", b="entrezB"):
    out = df.copy()
    out[a] = pd.to_numeric(out[a], errors="coerce")
    out[b] = pd.to_numeric(out[b], errors="coerce")

    out["e1"] = out[[a, b]].min(axis=1)
    out["e2"] = out[[a, b]].max(axis=1)

    return out.dropna(subset=["e1", "e2"])

df_cons = canonicalize_edges(df_all, "entrezA", "entrezB")
df_intact = canonicalize_edges(df_out, "entrezA", "entrezB")

cons_edges = df_cons[["e1", "e2"]].drop_duplicates()
intact_edges = df_intact[["e1", "e2"]].drop_duplicates()

cons_keys = df_cons[["e1", "e2"]].drop_duplicates()
df_overlap= df_intact.merge(cons_keys, on=["e1", "e2"], how="inner")
df_overlap = df_overlap.dropna(subset=["entrezA", "entrezB"]).copy()

df_overlap

,#ID(s) interactor A,ID(s) interactor B,Alt. ID(s) interactor A,Alt. ID(s) interactor B,Alias(es) interactor A,Alias(es) interactor B,Interaction detection method(s),Publication 1st author(s),Publication Identifier(s),Taxid interactor A,...,Identification method participant A,Identification method participant B,uniprotA,uniprotB,geneA,geneB,entrezA,entrezB,e1,e2
11,uniprotkb:Q9H2D6-5,uniprotkb:Q9H2D6-5,intact:EBI-1023227,intact:EBI-1023227,psi-mi:q9h2d6-5(display_long)|uniprotkb:TRIOBP...,psi-mi:q9h2d6-5(display_long)|uniprotkb:TRIOBP...,"psi-mi:""MI:0071""(molecular sieving)",Li et al. (2007),pubmed:17629495|imex:IM-19091,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0816""(molecular weight estimation b...","psi-mi:""MI:0816""(molecular weight estimation b...",Q9H2D6-5,Q9H2D6-5,TRIOBP,TRIOBP,11078.0,11078.0,11078.0,11078.0
12,uniprotkb:P40337-3,uniprotkb:Q16665,intact:EBI-301270|intact:EBI-8411390|intact:MI...,intact:EBI-447269|uniprotkb:Q53XP6|uniprotkb:Q...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,psi-mi:hif1a_human(display_long)|uniprotkb:ARN...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)",P40337-3,Q16665,VHL,HIF1A,7428.0,3091.0,3091.0,7428.0
13,uniprotkb:P40337-3,uniprotkb:Q16665,intact:EBI-301270|intact:EBI-8411390|intact:MI...,intact:EBI-447269|uniprotkb:Q53XP6|uniprotkb:Q...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,psi-mi:hif1a_human(display_long)|uniprotkb:ARN...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)",P40337-3,Q16665,VHL,HIF1A,7428.0,3091.0,3091.0,7428.0
14,uniprotkb:P40337-3,uniprotkb:Q16665,intact:EBI-301270|intact:EBI-8411390|intact:MI...,intact:EBI-447269|uniprotkb:Q53XP6|uniprotkb:Q...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,psi-mi:hif1a_human(display_long)|uniprotkb:ARN...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)",P40337-3,Q16665,VHL,HIF1A,7428.0,3091.0,3091.0,7428.0
15,uniprotkb:Q15369,uniprotkb:P40337-3,intact:EBI-301231|uniprotkb:Q567Q6|ensembl:ENS...,intact:EBI-301270|intact:EBI-8411390|intact:MI...,psi-mi:eloc_human(display_long)|uniprotkb:ELOC...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",Q15369,P40337-3,ELOC,VHL,6921.0,7428.0,6921.0,7428.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,uniprotkb:P04626,uniprotkb:Q99952-1,intact:EBI-641062|uniprotkb:Q14256|uniprotkb:Q...,intact:EBI-12739708|ensembl:ENSP00000175756.5,psi-mi:erbb2_human(display_long)|uniprotkb:p18...,psi-mi:q99952-1(display_long)|uniprotkb:Brain-...,"psi-mi:""MI:0007""(anti tag coimmunoprecipitation)",Wang et al. (2014),pubmed:25081058|imex:IM-25721,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0113""(western blot)","psi-mi:""MI:0113""(western blot)",P04626,Q99952-1,ERBB2,PTPN18,2064.0,26469.0,2064.0,26469.0
162,uniprotkb:Q99952-1,uniprotkb:P04626,intact:EBI-12739708|ensembl:ENSP00000175756.5,intact:EBI-641062|uniprotkb:Q14256|uniprotkb:Q...,psi-mi:q99952-1(display_long)|uniprotkb:Brain-...,psi-mi:erbb2_human(display_long)|uniprotkb:p18...,"psi-mi:""MI:0096""(pull down)",Wang et al. (2014),pubmed:25081058|imex:IM-25721,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0113""(western blot)","psi-mi:""MI:0113""(western blot)",Q99952-1,P04626,PTPN18,ERBB2,26469.0,2064.0,2064.0,26469.0
163,uniprotkb:Q99952-1,uniprotkb

In [ ]:
# only keep unqiue edges
tmp = df_overlap.copy()

tmp["_ia"] = tmp[["#ID(s) interactor A", "ID(s) interactor B"]].min(axis=1)
tmp["_ib"] = tmp[["#ID(s) interactor A", "ID(s) interactor B"]].max(axis=1)

df_overlap_norepeats = (
    tmp.drop_duplicates(subset=["_ia", "_ib"], keep="first")
       .drop(columns=["_ia", "_ib"])
       .copy()
)
df_overlap_norepeats

,#ID(s) interactor A,ID(s) interactor B,Alt. ID(s) interactor A,Alt. ID(s) interactor B,Alias(es) interactor A,Alias(es) interactor B,Interaction detection method(s),Publication 1st author(s),Publication Identifier(s),Taxid interactor A,...,Identification method participant A,Identification method participant B,uniprotA,uniprotB,geneA,geneB,entrezA,entrezB,e1,e2
11,uniprotkb:Q9H2D6-5,uniprotkb:Q9H2D6-5,intact:EBI-1023227,intact:EBI-1023227,psi-mi:q9h2d6-5(display_long)|uniprotkb:TRIOBP...,psi-mi:q9h2d6-5(display_long)|uniprotkb:TRIOBP...,"psi-mi:""MI:0071""(molecular sieving)",Li et al. (2007),pubmed:17629495|imex:IM-19091,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0816""(molecular weight estimation b...","psi-mi:""MI:0816""(molecular weight estimation b...",Q9H2D6-5,Q9H2D6-5,TRIOBP,TRIOBP,11078.0,11078.0,11078.0,11078.0
12,uniprotkb:P40337-3,uniprotkb:Q16665,intact:EBI-301270|intact:EBI-8411390|intact:MI...,intact:EBI-447269|uniprotkb:Q53XP6|uniprotkb:Q...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,psi-mi:hif1a_human(display_long)|uniprotkb:ARN...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)",P40337-3,Q16665,VHL,HIF1A,7428.0,3091.0,3091.0,7428.0
15,uniprotkb:Q15369,uniprotkb:P40337-3,intact:EBI-301231|uniprotkb:Q567Q6|ensembl:ENS...,intact:EBI-301270|intact:EBI-8411390|intact:MI...,psi-mi:eloc_human(display_long)|uniprotkb:ELOC...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",Q15369,P40337-3,ELOC,VHL,6921.0,7428.0,6921.0,7428.0
17,uniprotkb:P40337-3,uniprotkb:Q99814,intact:EBI-301270|intact:EBI-8411390|intact:MI...,intact:EBI-447470|uniprotkb:Q86VA2|uniprotkb:Q...,psi-mi:p40337-3(display_long)|uniprotkb:VHL19(...,psi-mi:epas1_human(display_long)|uniprotkb:EPA...,"psi-mi:""MI:0018""(two hybrid)",Bex et al. (2007),pubmed:17986458,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",P40337-3,Q99814,VHL,EPAS1,7428.0,2034.0,2034.0,7428.0
18,uniprotkb:P41134,uniprotkb:P15923-1,intact:EBI-1215527|uniprotkb:O00651|uniprotkb:...,intact:EBI-769645|ensembl:ENSP00000262965.5,psi-mi:id1_human(display_long)|uniprotkb:ID1(g...,psi-mi:p15923-1(display_long)|uniprotkb:PAN-2(...,"psi-mi:""MI:0018""(two hybrid)",Terai et al. (2000),pubmed:10915743|imex:IM-19434,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",P41134,P15923-1,ID1,TCF3,3397.0,6929.0,3397.0,6929.0
19,uniprotkb:Q02363,uniprotkb:P15923-1,intact:EBI-713450|ensembl:ENSP00000234091.4|en...,intact:EBI-769645|ensembl:ENSP00000262965.5,psi-mi:id2_human(display_long)|uniprotkb:Inhib...,psi-mi:p15923-1(display_long)|uniprotkb:PAN-2(...,"psi-mi:""MI:0018""(two hybrid)",Terai et al. (2000),pubmed:10915743|imex:IM-19434,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",Q02363,P15923-1,ID2,TCF3,3398.0,6929.0,3398.0,6929.0
20,uniprotkb:P15923-1,uniprotkb:Q02535,intact:EBI-769645|ensembl:ENSP00000262965.5,intact:EBI-1387094|uniprotkb:O75641|uniprotkb:...,psi-mi:p15923-1(display_long)|uniprotkb:PAN-2(...,psi-mi:id3_human(display_long)|uniprotkb:ID3(g...,"psi-mi:""MI:0018""(two hybrid)",Terai et al. (2000),pubmed:10915743|imex:IM-19434,taxid:9606(human)|taxid:9606(Homo sapiens),...,"psi-mi:""MI:0078""(nucleotide sequence identific...","psi-mi:""MI:0078""(nucleotide sequence identific...",P15923-1,Q02535,TCF3,ID3,6929.0,3399.0,3399.0,6929.0
22,uniprotkb:P15923-1,uniprotkb:P15172,intact:

In [ ]:
df_overlap_norepeats.to_csv("prosate_intact_overlap.tsv", sep="\t", index=False)

## Hypergeometric test for enrichment of isoform-specific PPIs in Splitpea rewired network

In [ ]:
from scipy.stats import hypergeom

N = 787570  # possible edges from reference PPI (splitpea/src/reference/human_ppi_0.5.dat)
K = 730     # isoform-specific PPIs in IntAct
n = 17012   # edges in Splitpea rewired network
k = 29      # overlapping edges between Splitpea rewired network and IntAct isoform-specific PPIs

p_over = hypergeom.sf(k-1, N, K, n)  
print(p_over)

0.0015682229616878801
